# Sourdough Assistant — Arize Eval Notebook

This notebook drives the full Arize evaluation workflow:
1. Pull traces from Phoenix for a given experiment
2. Run six LLM-as-judge evaluators against the traces
3. Upload results back to Arize as an experiment
4. Compare iterations (baseline → iter1 → iter2)

## Setup
```
pip install arize-phoenix openai anthropic pandas
```

Set `PHOENIX_API_KEY`, `ANTHROPIC_API_KEY`, and `PHOENIX_COLLECTOR_ENDPOINT` in your environment.

In [ ]:
import os
import json
import pandas as pd
from anthropic import Anthropic

ANTHROPIC_API_KEY = os.environ['ANTHROPIC_API_KEY']
PHOENIX_API_KEY = os.environ.get('PHOENIX_API_KEY', '')
PHOENIX_BASE_URL = os.environ.get('PHOENIX_BASE_URL', 'https://app.phoenix.arize.com')

client = Anthropic(api_key=ANTHROPIC_API_KEY)
print('Setup complete')

In [ ]:
# Load golden dataset
with open('golden-dataset.json') as f:
    golden = json.load(f)

df_golden = pd.DataFrame(golden)
print(f'Loaded {len(df_golden)} scenarios')
df_golden[['id', 'mode', 'difficulty']].head(10)

## Run the Agent Against Golden Scenarios

This cell calls the local dev server for each scenario and collects the agent's responses.
Make sure `npm run dev` is running at `http://localhost:3000`.

In [ ]:
import requests
import time

API_URL = 'http://localhost:3000/api/chat'

def run_scenario(scenario: dict, model_id: str = 'anthropic/claude-sonnet-4-6') -> dict:
    """Run a golden scenario against the agent and return the full response."""
    messages = []
    full_response = ''
    tool_calls = []
    
    for user_msg in scenario['user_messages']:
        messages.append({
            'id': f'msg_{len(messages)}',
            'role': 'user',
            'parts': [{'type': 'text', 'text': user_msg}]
        })
        
        response = requests.post(
            API_URL,
            json={'messages': messages, 'id': scenario['id']},
            headers={'Content-Type': 'application/json', 'x-model-id': model_id},
            stream=True,
            timeout=60
        )
        
        # Parse SSE stream
        assistant_text = ''
        for line in response.iter_lines():
            if line:
                decoded = line.decode('utf-8')
                if decoded.startswith('0:'):  # text delta
                    try:
                        text = json.loads(decoded[2:])
                        assistant_text += text
                    except:
                        pass
        
        full_response = assistant_text
        messages.append({
            'id': f'msg_{len(messages)}',
            'role': 'assistant',
            'parts': [{'type': 'text', 'text': assistant_text}]
        })
        time.sleep(0.5)
    
    return {
        'scenario_id': scenario['id'],
        'mode': scenario['mode'],
        'difficulty': scenario['difficulty'],
        'user_messages': scenario['user_messages'],
        'agent_response': full_response,
        'expected_root_cause': scenario.get('expected_root_cause_id'),
        'expected_recipe': scenario.get('expected_recipe_id'),
        'expected_symptoms': scenario.get('expected_symptoms', []),
        'model_id': model_id,
    }

print('run_scenario() defined')

In [ ]:
# Run all scenarios — this takes ~5-10 min
results = []
for i, scenario in enumerate(golden):
    print(f'Running {i+1}/{len(golden)}: {scenario["id"]}...')
    try:
        result = run_scenario(scenario)
        results.append(result)
    except Exception as e:
        print(f'  ERROR: {e}')
        results.append({'scenario_id': scenario['id'], 'error': str(e)})

df_results = pd.DataFrame(results)
print(f'\nCompleted {len(df_results)} scenarios')
df_results.head(3)

## Evaluators

Six LLM-as-judge evaluators, each using Claude at temperature=0.

In [ ]:
def llm_judge(prompt: str) -> tuple[str, str]:
    """Call Claude as judge. Returns (score: PASS/FAIL/NA, reasoning)."""
    resp = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=256,
        temperature=0,
        messages=[{'role': 'user', 'content': prompt}]
    )
    text = resp.content[0].text.strip()
    # Extract score from first line
    first_line = text.split('\n')[0].upper()
    if 'PASS' in first_line:
        score = 'PASS'
    elif 'FAIL' in first_line:
        score = 'FAIL'
    else:
        score = 'NA'
    return score, text

print('llm_judge() defined')

In [ ]:
def eval_mode_detection(row) -> tuple[str, str]:
    """Did the agent correctly identify troubleshooting vs recipe mode?"""
    prompt = f"""You are evaluating an AI sourdough assistant.

User input: {row['user_messages']}
Expected mode: {row['mode']}
Agent response: {row['agent_response'][:800]}

Did the agent correctly identify the user's intent as '{row['mode']}' mode?
- For 'troubleshooting': the agent should ask about symptoms or diagnose a problem.
- For 'recipe': the agent should offer recipe guidance.

Respond with PASS or FAIL on the first line, then one sentence of reasoning."""
    return llm_judge(prompt)


def eval_diagnostic_accuracy(row) -> tuple[str, str]:
    """Did the agent reach the correct root cause? (troubleshooting mode only)"""
    if row['mode'] != 'troubleshooting' or not row.get('expected_root_cause'):
        return 'NA', 'Not a troubleshooting scenario'
    prompt = f"""You are evaluating an AI sourdough troubleshooting assistant.

User problem: {row['user_messages']}
Expected root cause ID: {row['expected_root_cause']}
Agent response: {row['agent_response'][:1000]}

Did the agent identify the correct root cause that matches '{row['expected_root_cause']}'?
The agent doesn't need to cite the ID — it just needs to identify the correct underlying problem.

Respond with PASS or FAIL on the first line, then one sentence of reasoning."""
    return llm_judge(prompt)


def eval_recipe_accuracy(row) -> tuple[str, str]:
    """Did the agent retrieve the right recipe and follow its steps? (recipe mode only)"""
    if row['mode'] != 'recipe' or not row.get('expected_recipe'):
        return 'NA', 'Not a recipe scenario'
    prompt = f"""You are evaluating an AI sourdough recipe assistant.

User request: {row['user_messages']}
Expected recipe: {row['expected_recipe']}
Agent response: {row['agent_response'][:1000]}

Did the agent identify and start walking through the correct recipe '{row['expected_recipe']}'?

Respond with PASS or FAIL on the first line, then one sentence of reasoning."""
    return llm_judge(prompt)


def eval_clarification_quality(row) -> tuple[str, str]:
    """Did the agent gather enough symptoms or clarify intent before diagnosing/recommending?"""
    if row['mode'] == 'troubleshooting':
        expected_symptoms = row.get('expected_symptoms', [])
        prompt = f"""You are evaluating an AI sourdough troubleshooting assistant.

User input: {row['user_messages']}
Expected symptoms to identify: {expected_symptoms}
Agent response: {row['agent_response'][:1000]}

Did the agent:
1. Record or acknowledge the symptoms mentioned? (via recordSymptom calls or explicit acknowledgment)
2. Ask for clarification when information was ambiguous?
3. NOT jump to diagnosis without first acknowledging what the user told them?

Respond with PASS or FAIL on the first line, then one sentence of reasoning."""
    else:
        prompt = f"""You are evaluating an AI sourdough recipe assistant.

User input: {row['user_messages']}
Agent response: {row['agent_response'][:1000]}

Did the agent ask clarifying questions when the recipe request was vague, OR proceed directly with the right recipe when the request was specific?

Respond with PASS or FAIL on the first line, then one sentence of reasoning."""
    return llm_judge(prompt)


def eval_faithfulness(row) -> tuple[str, str]:
    """Did the agent cite causes/fixes not present in general sourdough knowledge, or hallucinate?"""
    prompt = f"""You are evaluating an AI sourdough assistant for hallucination.

User input: {row['user_messages']}
Agent response: {row['agent_response'][:1000]}

Check if the agent:
1. Made up specific numbers/temperatures/ratios that seem implausible
2. Contradicted itself
3. Gave advice that is factually wrong for sourdough baking
4. For safety scenarios (pink/orange starter, mold): did it recommend discarding? FAIL if it suggested feeding through or saving.

Respond with PASS (no hallucination) or FAIL (hallucination detected) on the first line, then one sentence explaining what was wrong (or confirming it was correct)."""
    return llm_judge(prompt)


def eval_conciseness(row) -> tuple[str, str]:
    """Did the agent avoid excessive rambling?"""
    response_len = len(row['agent_response'])
    prompt = f"""You are evaluating an AI sourdough assistant for conciseness.

User input: {row['user_messages']}
Agent response ({response_len} chars): {row['agent_response'][:1200]}

Is the response appropriately concise? A good response:
- Answers the question directly without excessive preamble
- Doesn't repeat itself
- Is long enough to be useful but not padded with unnecessary caveats
- Doesn't ask more than 2 clarifying questions at once

Respond with PASS (good length and directness) or FAIL (too long, repetitive, or rambling) on the first line, then one sentence of reasoning."""
    return llm_judge(prompt)


print('All evaluators defined')

In [ ]:
# Run all evaluators against all results
eval_fns = {
    'mode_detection': eval_mode_detection,
    'diagnostic_accuracy': eval_diagnostic_accuracy,
    'recipe_accuracy': eval_recipe_accuracy,
    'clarification_quality': eval_clarification_quality,
    'faithfulness': eval_faithfulness,
    'conciseness': eval_conciseness,
}

for eval_name, eval_fn in eval_fns.items():
    scores = []
    reasonings = []
    for _, row in df_results.iterrows():
        try:
            score, reasoning = eval_fn(row)
        except Exception as e:
            score, reasoning = 'ERROR', str(e)
        scores.append(score)
        reasonings.append(reasoning)
    df_results[f'{eval_name}_score'] = scores
    df_results[f'{eval_name}_reasoning'] = reasonings
    pass_rate = sum(1 for s in scores if s == 'PASS') / max(1, sum(1 for s in scores if s != 'NA'))
    print(f'{eval_name}: {pass_rate:.1%} pass rate')

print('\nEval run complete')

In [ ]:
# Summary table
score_cols = [c for c in df_results.columns if c.endswith('_score')]
summary = {}
for col in score_cols:
    evaluator = col.replace('_score', '')
    scores = df_results[col]
    eligible = scores[scores != 'NA']
    pass_rate = (eligible == 'PASS').sum() / max(1, len(eligible))
    summary[evaluator] = {
        'pass_rate': f'{pass_rate:.1%}',
        'passes': (eligible == 'PASS').sum(),
        'fails': (eligible == 'FAIL').sum(),
        'na': (scores == 'NA').sum(),
    }

pd.DataFrame(summary).T

In [ ]:
# Save results for comparison
df_results.to_csv('eval_results_baseline.csv', index=False)
print('Saved eval_results_baseline.csv')

## Upload to Arize Phoenix

Upload the golden dataset and eval results as a versioned Arize experiment.

In [ ]:
import phoenix as px
from phoenix.experiments import run_experiment
from phoenix.experiments.types import EvaluationResult

# Initialize Phoenix client
px_client = px.Client(
    endpoint=PHOENIX_BASE_URL,
    api_key=PHOENIX_API_KEY,
)

# Upload golden dataset
dataset = px_client.upload_dataset(
    dataframe=df_golden,
    dataset_name='sourdough-golden-dataset',
    input_keys=['user_messages', 'mode', 'difficulty'],
    output_keys=['expected_root_cause_id', 'expected_recipe_id', 'expected_symptoms'],
    metadata_keys=['id', 'notes'],
)
print(f'Dataset uploaded: {dataset.id}')

In [ ]:
# Example: Compare two runs (e.g., baseline vs iter1)
# Load both result CSVs and compare pass rates

import os

def load_and_summarize(filename: str, label: str):
    if not os.path.exists(filename):
        print(f'{filename} not found')
        return None
    df = pd.read_csv(filename)
    score_cols = [c for c in df.columns if c.endswith('_score')]
    rows = []
    for col in score_cols:
        evaluator = col.replace('_score', '')
        scores = df[col]
        eligible = scores[scores != 'NA']
        pass_rate = (eligible == 'PASS').sum() / max(1, len(eligible))
        rows.append({'evaluator': evaluator, label: f'{pass_rate:.1%}'})
    return pd.DataFrame(rows).set_index('evaluator')

baseline = load_and_summarize('eval_results_baseline.csv', 'baseline')
iter1 = load_and_summarize('eval_results_iter1.csv', 'iter1')
iter2 = load_and_summarize('eval_results_iter2.csv', 'iter2')

frames = [f for f in [baseline, iter1, iter2] if f is not None]
if frames:
    comparison = pd.concat(frames, axis=1)
    print(comparison)
else:
    print('No result files found yet')

## Cross-Model Comparison

Run the final iter2 prompt across 5 OpenRouter models and compare eval scores.

In [ ]:
COMPARISON_MODELS = [
    "anthropic/claude-sonnet-4-6",
    "anthropic/claude-haiku-4-5",
    "google/gemini-flash-2.0",
    "meta-llama/llama-3.3-70b-instruct",
    "mistralai/mistral-small",
]

model_results = {}

for model_id in COMPARISON_MODELS:
    print(f"\nRunning {model_id}...")
    m_results = []
    for i, scenario in enumerate(golden[:10]):  # subset for speed — expand for full run
        try:
            result = run_scenario(scenario, model_id=model_id)
            m_results.append(result)
        except Exception as e:
            print(f"  ERROR: {e}")
    model_results[model_id] = pd.DataFrame(m_results)
    print(f"  Completed {len(m_results)} scenarios")

print("Model comparison runs complete")


In [ ]:
# Score each model
model_summaries = {}

for model_id, df_m in model_results.items():
    for eval_name, eval_fn in eval_fns.items():
        scores = []
        for _, row in df_m.iterrows():
            try:
                score, _ = eval_fn(row)
                scores.append(score)
            except:
                scores.append("ERROR")
        df_m[f"{eval_name}_score"] = scores

    score_cols = [c for c in df_m.columns if c.endswith("_score")]
    row_data = {}
    for col in score_cols:
        evaluator = col.replace("_score", "")
        eligible = df_m[col][df_m[col] != "NA"]
        pass_rate = (eligible == "PASS").sum() / max(1, len(eligible))
        row_data[evaluator] = f"{pass_rate:.1%}"
    model_summaries[model_id.split("/")[-1]] = row_data

pd.DataFrame(model_summaries).T


In [ ]:
# Save model comparison
pd.DataFrame(model_summaries).T.to_csv("eval_model_comparison.csv")
print("Saved eval_model_comparison.csv")
